# Khóa Luận CNTT-KLCN140: Hybrid PO–GA (Fitness 4 Thành Phần) & Đánh Giá trên ResNet-18, MobileNetV3-Small, DenseNet-121
$$F(X) = 0.15\,\text{Div}(X) + 0.45\,\text{Cov}(X) + 0.20\,\text{Bal}(X) + 0.20\,\text{Com}(X)$$
**Hướng dẫn chạy trên Kaggle:**
1. Bật GPU ở menu bên phải: **Settings -> Accelerator -> GPU T4 x2 (hoặc P100)** và bật **Internet -> On**.
2. Nhấn **Add Input** và thêm Dataset ảnh: `https://www.kaggle.com/datasets/puneet6060/intel-image-classification`
3. Chạy lần lượt 3 ô (Cells) bên dưới (Cell 1 sẽ tự động giải nén từ Input nếu có `.zip`, hoặc tự động `git clone -b jaxy` từ GitHub).

In [ ]:
import os, shutil, zipfile, subprocess, json, base64, zlib

WORK_DIR = '/kaggle/working/KhoaLuanDatasetReduction'
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.zip') and 'KhoaLuanDatasetReduction' in f:
            zip_path = os.path.join(root, f)
            print(f'Giải nén từ: {zip_path}')
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(WORK_DIR)
            break
    if 'run_poga.py' in files and 'train.py' in files:
        print(f'Sao chép từ: {root}')
        shutil.copytree(root, WORK_DIR, dirs_exist_ok=True)

# Tự động ghi đè toàn bộ mã nguồn Python mới nhất (hoạt động cả khi GitHub Private hoặc dùng zip cũ)
CODE_B64 = '...'
patches = json.loads(zlib.decompress(base64.b64decode(CODE_B64)).decode('utf-8'))
for rel_path, content in patches.items():
    full_path = os.path.join(WORK_DIR, rel_path)
    os.makedirs(os.path.dirname(full_path) or WORK_DIR, exist_ok=True)
    with open(full_path, 'w', encoding='utf-8') as f:
        f.write(content)
print(f'Đã nạp thành công {len(patches)} file mã nguồn mới nhất!')

%cd /kaggle/working/KhoaLuanDatasetReduction
os.environ['PYTHONPATH'] = '/kaggle/working/KhoaLuanDatasetReduction'
!ls -la

In [ ]:
# Chạy thực nghiệm HÀM FITNESS 4 THÀNH PHẦN (F = 0.15*Div + 0.45*Cov + 0.20*Bal + 0.20*Com)
# So sánh toàn bộ 10 phương pháp (S1-S4, P1, C1, GA, PO, Random, Stratified) trên 3 mô hình (ResNet-18, MobileNetV3, DenseNet-121)
!PYTHONPATH=/kaggle/working/KhoaLuanDatasetReduction python run_poga.py --method all --model all --epochs 5 --batch_size 64 --seeds 42 123 456

In [ ]:
# Nén toàn bộ thư mục kết quả (/kaggle/working/outputs) thành 1 file zip để tải về máy
!zip -r /kaggle/working/ket_qua_khoa_luan_poga.zip /kaggle/working/outputs
print('Hoàn tất! Tải file /kaggle/working/ket_qua_khoa_luan_poga.zip ở mục Output bên phải.')